In [ ]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

#load dataset 
dataset = pd.read_excel("dataset.xlsx", index_col=0, na_values=["??","????", " "])

data = dataset.copy()
# print(data.describe())

soft_skills_map = {'Poor': 0, 'Average': 1, 'Good': 2, 'Excellent': 3}
coding_map = {'Beginner': 0, 'Intermediate': 1, 'Advanced': 2}
communication_skills_map = {'Poor': 0 , 'Average': 1 , 'Good' : 2 , 'Excellent' : 3 } 
data['coding_skills'] = data['coding_skills'].map(coding_map)
data['soft_skills'] = data['soft_skills'].map(soft_skills_map)
data['communication_skills'] = data['communication_skills'].map(communication_skills_map)
data['gender'] = data['gender'].map({'Male': 1, 'Female': 0})
data['degree'] = data['degree'].map({'BCA': 0 , 'B.Sc' : 1 , 'MCA' : 2})
data['branch'] = data['branch'].map({'EEE' : 0 , 'CE' : 1 , 'ECE' : 2 , 'ME' : 3 , 'CSE' : 4})


data['projects']= data['projects'].fillna(data['projects'].mean().round(0))
data['age'] = data['age'].fillna(data['age'].mean().round(0))
data['soft_skills'] = data['soft_skills'].fillna(data['soft_skills'].mode()[0])
data['communication_skills'] = data['communication_skills'].fillna(data['communication_skills'].mode()[0])



# data['soft_skills']= data['soft_skills'].fillna(data['soft_skills'].mean())
# print( data.isnull().sum()) # we have 3 mising value in communication_skills and 2 soft_skills 

# print(data['soft_skills'].mode())

#cheack tarket value 
# Input and output
X = data.drop(
    columns=["status", "salary_package_lpa",
             "technical_interview_mock",
             "hackathon_leetcode_problem"],
    errors="ignore"
)

y = data["status"]


# Separate text and number columns
cat_cols = X.select_dtypes(include=["object", "str"]).columns
num_cols = X.select_dtypes(include=["number"]).columns

# Preprocessing
prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

# Model
model = Pipeline([
    ("prep", prep),
    ("clf", RandomForestClassifier(n_estimators=100,random_state=42))
])

# Train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, pred) * 100, 2), "%")
"""age = input("Enter Student Age ")
degree = input("Enter Student Degree like (BCA,B.Sc,MCA) ")
branch = input("Enter Student Branch like (CSE,CE,CSE,ME) ")

cgpa = input("Enter Student CGPA ")
backlog = input("Enter Student Backlog ")
intership = input("Enter Student Total  Intership certificate ")
coding_skills = input("Enter Student Coding Skill level like (Intermediate , Advanced)")
aptitude_score = input("Enter Student aptitude score number ")
communication_skills = input("Enter Student communication skills like (Poor , Good , Average , Excellent)")
projects = input("Enter Student Total number of projects ")
soft_skills = input("Enter Student Soft skill like (Poor , Good , Average , Excellent)")
"""


def get_number(message, min_value, max_value,integer=False):
    while True:
        try:
            value = input(message).strip().lower()
            
            if integer:
                number = int(value)
            else:
                number =float(value)
                
            if not min_value <= number <= max_value:
                print(f"Please Enter a value Between {min_value} and {max_value}")
                continue
            return number
        except ValueError:
            print("Invalid input ! Please Enter A valid number ")
            
            
def get_choices(message, choices):
    choices_lower = {choice.strip().lower(): choice.strip() for choice in choices}
    
    while True:
        value = input(message).strip().lower()
        
        if value in choices_lower:
            return choices_lower[value]
        print("Invalid Choices ! Choice from ", ", ".join(choices))
        
        
age = get_number("Enter Age : ", 18,40,integer=True)

degree = get_choices(
    "Enter Degree (BCA , B.Sc, MCA):",
    ["BCA","B.Sc","MCA"])
branch = get_choices(
    "Enter Branch (CSE, CE, ME):",
    ["CSE","CE", "ME"])

cgpa = get_number(
    "Enter CGPA ",
    0,10,
    integer=True
)

backlog = get_number(
    "Enter All Backlogs ",
    0,20,
    integer=True
)

intership= get_number(
    "Enter internship certificaes :",
    0,
    20,
    integer=True
)

coding_skills = get_choices(
    "Enter Coding Skill (Intermediate , Advanced):",
    ["Advanced","Intermediate "]
)


projects = get_number(
    "Enter All Project NUmber ",
    0,
    100,
    integer=True
)

soft_skills= get_choices(
    "Enter Soft Skill (Poor , Good , Average , Excellent):",
    ["Poor ", "Good" , "Average" , "Excellent"]
)

communication_skills = get_choices(
    "Enter Communication skill (Poor , Good , Average , Excellent):",
    ["Poor" , "Good" , "Average" , "Excellent"]
)

aptitude_score = get_number(
    "Enter aptitude_score ",
    0,
    100,
    integer=True
) 


# New student input
student = pd.DataFrame([{
    "age": float(age),
    "degree": degree,
    "branch": branch,
    "cgpa": float(cgpa),
    "backlogs": int(backlog),
    "internship_certificate": int(intership),
    "coding_skills": coding_skills,
    "aptitude_score": float(aptitude_score),
    "communication_skills": communication_skills,
    "projects": float(projects),
    "soft_skills": soft_skills
}])

result = model.predict(student)[0]
print("Predicted status:", result)
if result == 1:
    print("Predicted Status: Placed")
else:
    print("Predicted Status: Not Placed")
    
cm = confusion_matrix(y_test, pred)
print("\nConfusion Matrix:")
print(cm)

